# Tạo partition Non-IID Dirichlet cho MNIST

Notebook này tương đương với `partition_dirichlet.py`. Các tham số chạy được đặt trong cell cấu hình.

In [ ]:
import csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets

In [ ]:
# Cấu hình chạy
ALPHA = 0.5
PARTITION_TYPE = 'dirichlet'  # 'dirichlet' hoặc 'noniid2'
NUM_CLIENTS = 25
SEED = 42
MIN_SAMPLES = 10
OVERWRITE = False
NO_PLOT = False
PROJECT_ROOT = None  # None: tự động xác định; hoặc đặt Path('/đường/dẫn/dự/án')

In [ ]:
def get_default_project_root() -> Path:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        project_root = Path('/content/drive/MyDrive/FLasIRL(new)')
    except ImportError:
        project_root = Path.cwd() / 'FLasIRL(new)' if not Path('/content').exists() else Path('/content/FLasIRL(new)')

    if not project_root.parent.exists():
        project_root = Path.cwd()
    return project_root


def generate_dirichlet_partition(
    train_labels: torch.Tensor,
    num_clients: int = 25,
    alpha: float = 0.5,
    seed: int = 42,
    min_samples: int = 10,
    max_retries: int = 100,
) -> tuple[list[torch.Tensor], torch.Tensor]:
    labels_np = train_labels.cpu().numpy()
    num_samples = len(labels_np)
    classes = np.unique(labels_np)
    num_classes = len(classes)

    current_seed = seed
    for attempt in range(max_retries):
        rng = np.random.default_rng(current_seed)
        client_sample_indices = [[] for _ in range(num_clients)]

        for c in classes:
            idx_c = np.where(labels_np == c)[0]
            rng.shuffle(idx_c)
            proportions = rng.dirichlet(np.repeat(alpha, num_clients))
            counts = rng.multinomial(len(idx_c), proportions)

            start = 0
            for client_id, count in enumerate(counts):
                if count > 0:
                    client_sample_indices[client_id].extend(idx_c[start : start + count])
                    start += count

        client_sizes = [len(indices) for indices in client_sample_indices]
        if min(client_sizes) >= min_samples:
            client_indices = []
            client_class_counts = torch.zeros((num_clients, num_classes), dtype=torch.long)

            for client_id, indices in enumerate(client_sample_indices):
                indices_arr = np.array(indices, dtype=np.int64)
                rng.shuffle(indices_arr)
                t_indices = torch.tensor(indices_arr, dtype=torch.long)
                client_indices.append(t_indices)
                client_class_counts[client_id] = torch.bincount(
                    train_labels[t_indices], minlength=num_classes
                )

            assert sum(len(x) for x in client_indices) == num_samples
            assert len(torch.cat(client_indices).unique()) == num_samples
            return client_indices, client_class_counts

        current_seed += 1

    raise RuntimeError(
        f'Không thể tạo partition Dirichlet với alpha={alpha} và min_samples={min_samples} '
        f'sau {max_retries} lần thử.'
    )

In [ ]:
def generate_two_class_partition(
    train_labels: torch.Tensor,
    num_clients: int = 25,
    seed: int = 42,
    min_samples: int = 10,
) -> tuple[list[torch.Tensor], torch.Tensor]:
    # Chia dữ liệu sao cho mỗi client chỉ nhận mẫu của đúng hai class.
    labels_np = train_labels.cpu().numpy()
    classes = np.unique(labels_np)
    num_classes = len(classes)
    rng = np.random.default_rng(seed)

    # Phân bố client vào các class theo vòng tròn, sau đó xáo trộn assignment.
    if num_classes < 2:
        raise ValueError('Cần ít nhất 2 class để tạo partition noniid2.')
    client_classes = np.array([
        [classes[(2 * i) % num_classes], classes[(2 * i + 1) % num_classes]]
        for i in range(num_clients)
    ])
    rng.shuffle(client_classes, axis=0)
    client_indices = [[] for _ in range(num_clients)]

    for class_id in classes:
        class_clients = np.where(client_classes == class_id)[0]
        class_indices = np.flatnonzero(labels_np == class_id)
        rng.shuffle(class_indices)
        for client_id, indices in zip(class_clients, np.array_split(class_indices, len(class_clients))):
            client_indices[client_id].extend(indices.tolist())

    client_sizes = [len(indices) for indices in client_indices]
    if min(client_sizes) < min_samples:
        raise RuntimeError(
            f'Partition noniid2 có client dưới min_samples={min_samples}: '
            f'min={min(client_sizes)}.'
        )

    client_class_counts = torch.zeros((num_clients, num_classes), dtype=torch.long)
    client_index_tensors = []
    for client_id, indices in enumerate(client_indices):
        indices_arr = np.asarray(indices, dtype=np.int64)
        rng.shuffle(indices_arr)
        t_indices = torch.tensor(indices_arr, dtype=torch.long)
        client_index_tensors.append(t_indices)
        client_class_counts[client_id] = torch.bincount(
            train_labels[t_indices], minlength=num_classes
        )

    assert sum(len(x) for x in client_index_tensors) == len(train_labels)
    assert len(torch.cat(client_index_tensors).unique()) == len(train_labels)
    assert torch.all((client_class_counts > 0).sum(dim=1) == 2)
    return client_index_tensors, client_class_counts

In [ ]:
def plot_and_save_distribution(
    client_class_counts: torch.Tensor,
    num_clients: int,
    alpha: float,
    seed: int,
    chart_path: Path,
) -> None:
    num_classes = client_class_counts.shape[1]
    client_sizes = client_class_counts.sum(dim=1).cpu().numpy()

    plt.figure(figsize=(14, 6))
    bottom = np.zeros(num_clients)
    colors = plt.cm.tab10(np.linspace(0, 1, num_classes))

    for cls in range(num_classes):
        values = client_class_counts[:, cls].cpu().numpy()
        plt.bar(
            np.arange(num_clients), values, bottom=bottom, label=f'Class {cls}',
            color=colors[cls], edgecolor='black', linewidth=0.5,
        )
        bottom += values

    plt.xlabel('Client ID', fontsize=12)
    plt.ylabel('Number of Samples', fontsize=12)
    plt.title(
        f'Non-IID Class Distribution (K={num_clients}, Seed={seed})\n'
        f'Min Samples: {client_sizes.min()}, Max: {client_sizes.max()}, Mean: {client_sizes.mean():.1f}',
        fontsize=13,
    )
    plt.xticks(np.arange(num_clients), labels=[str(i) for i in range(num_clients)], fontsize=9)
    plt.legend(ncol=10, loc='upper center', bbox_to_anchor=(0.5, -0.15), frameon=True)
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()

    chart_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Biểu đồ phân bố đã được lưu tại: {chart_path}')

In [ ]:
# Xác định thư mục dữ liệu và đường dẫn output
project_root = Path(PROJECT_ROOT) if PROJECT_ROOT is not None else get_default_project_root()
data_dir = project_root / 'data' / 'mnist'
partition_dir = data_dir / 'partitions' / 'Dirichlet'
data_dir.mkdir(parents=True, exist_ok=True)
partition_dir.mkdir(parents=True, exist_ok=True)

if PARTITION_TYPE not in {'dirichlet', 'noniid2'}:
    raise ValueError("PARTITION_TYPE phải là 'dirichlet' hoặc 'noniid2'.")

if PARTITION_TYPE == 'noniid2':
    partition_filename = f'noniid2_{NUM_CLIENTS}_seed{SEED}.pt'
    chart_filename = f'partition_noniid2_{NUM_CLIENTS}_clients_seed{SEED}.png'
else:
    partition_filename = f'dirichlet_alpha{ALPHA}_{NUM_CLIENTS}_seed{SEED}.pt'
    chart_filename = f'partition_dirichlet_alpha{ALPHA}_{NUM_CLIENTS}_clients_seed{SEED}.png'
partition_path = partition_dir / partition_filename
chart_path = partition_dir / chart_filename

if partition_path.exists() and not OVERWRITE:
    print(f'File partition đã tồn tại: {partition_path}')
    print('Đặt OVERWRITE = True nếu muốn tạo lại.')
else:
    print(f'Đang tải dữ liệu MNIST từ: {data_dir} ...')
    train_ds = datasets.MNIST(data_dir, train=True, download=True)
    train_labels = train_ds.targets.long()
    print(f'Đã tải {len(train_labels)} mẫu huấn luyện với 10 class.')

    if PARTITION_TYPE == 'noniid2':
        print(
            f'Đang tạo partition noniid2: clients={NUM_CLIENTS}, '
            f'seed={SEED} ...'
        )
        client_indices, client_class_counts = generate_two_class_partition(
            train_labels=train_labels,
            num_clients=NUM_CLIENTS,
            seed=SEED,
            min_samples=MIN_SAMPLES,
        )
        distribution = 'noniid2'
    else:
        print(
            f'Đang tạo partition Non-IID Dirichlet: alpha={ALPHA}, '
            f'clients={NUM_CLIENTS}, seed={SEED} ...'
        )
        client_indices, client_class_counts = generate_dirichlet_partition(
            train_labels=train_labels,
            num_clients=NUM_CLIENTS,
            alpha=ALPHA,
            seed=SEED,
            min_samples=MIN_SAMPLES,
        )
        distribution = 'dirichlet'

    payload = {
        'client_indices': client_indices,
        'seed': SEED,
        'num_clients': NUM_CLIENTS,
        'alpha': ALPHA,
        'distribution': distribution,
        'min_samples': MIN_SAMPLES,
        'class_counts': client_class_counts,
        'client_sizes': [len(x) for x in client_indices],
    }
    torch.save(payload, partition_path)
    print(f'Đã lưu partition thành công tại: {partition_path}')

    client_sizes = [len(x) for x in client_indices]
    print(f'\n--- THỐNG KÊ PARTITION {PARTITION_TYPE.upper()} ---')
    print(f'Số client: {NUM_CLIENTS}')
    print(f'Tham số alpha: {ALPHA}')
    print(f'Tổng số mẫu: {sum(client_sizes)}')
    print(f'Mẫu ít nhất của 1 client: {min(client_sizes)}')
    print(f'Mẫu nhiều nhất của 1 client: {max(client_sizes)}')
    print(f'Trung bình mỗi client: {np.mean(client_sizes):.1f} (std: {np.std(client_sizes):.1f})')
    print('Kích thước các client:', client_sizes)

    if not NO_PLOT:
        plot_and_save_distribution(
            client_class_counts=client_class_counts,
            num_clients=NUM_CLIENTS,
            alpha=ALPHA,
            seed=SEED,
            chart_path=chart_path,
        )

In [ ]:
# Lưu số lượng mẫu của từng class cho mỗi client thành CSV trên Drive.
partition = torch.load(partition_path, map_location='cpu')
client_class_counts = partition['class_counts']
counts_csv_path = partition_path.with_name(partition_path.stem + '_class_counts.csv')
num_classes = client_class_counts.shape[1]
with counts_csv_path.open('w', newline='', encoding='utf-8') as handle:
    fieldnames = ['client_id', 'total_samples'] + [f'class_{class_id}' for class_id in range(num_classes)]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for client_id, counts in enumerate(client_class_counts.tolist()):
        writer.writerow({
            'client_id': client_id,
            'total_samples': sum(counts),
            **{f'class_{class_id}': count for class_id, count in enumerate(counts)},
        })
print('CSV class counts đã được lưu tại:', counts_csv_path)